### This Notebook Covers Different Aspects of AI Fundamentals

In [ ]:
import os
from pathlib import Path
import chromadb
from sentence_transformers import SentenceTransformer
from langchain_openai import ChatOpenAI
from langchain_core.documents import Document
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)
from langchain_community.document_loaders import (
    PyMuPDFLoader
)
from langchain_community.vectorstores import Chroma
from langchain_huggingface import (
    HuggingFaceEmbeddings
)

In [132]:
#Initialize  LLM

llm = ChatOpenAI(
    model = "llama3.2:3b",
    api_key="ollama",
    base_url="http://localhost:11434/v1"

)

In [133]:
response = llm.invoke("What is the time?")

In [134]:
print(response)

content='I\'m not sure what time it is right now, as I\'m a large language model, I don\'t have real-time access to the current time. However, I can suggest ways for you to find out the current time.\n\nYou can check the current time by:\n\n1. Looking at a clock or watch\n2. Searching for "current time" on a search engine like Google\n3. Checking your phone\'s clock or calendar app\n4. Asking a voice assistant like Siri, Google Assistant, or Alexa\n\nIf you need help with anything else, feel free to ask!' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 118, 'prompt_tokens': 30, 'total_tokens': 148, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'llama3.2:3b', 'system_fingerprint': 'fp_ollama', 'id': 'chatcmpl-470', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019e9cc3-47e4-7c33-9a5a-504638a60769-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 30, 'o

In [135]:
print(type(response))

<class 'langchain_core.messages.ai.AIMessage'>


In [136]:
content = response.content 
print(content)

I'm not sure what time it is right now, as I'm a large language model, I don't have real-time access to the current time. However, I can suggest ways for you to find out the current time.

You can check the current time by:

1. Looking at a clock or watch
2. Searching for "current time" on a search engine like Google
3. Checking your phone's clock or calendar app
4. Asking a voice assistant like Siri, Google Assistant, or Alexa

If you need help with anything else, feel free to ask!


In [137]:
# Initialize PDF Reader
doc_dir = Path('..\\data\\pdf_files\\Arsenal Ebooks')
if doc_dir.exists():
    print('Path Found')
else :
    print('Path not Found')


Path Found


In [138]:
file_paths = doc_dir.iterdir()
for p in file_paths:
    print(f'Files Name : {p}')

Files Name : ..\data\pdf_files\Arsenal Ebooks\Arsene Wenger - Tom Oldfield.pdf
Files Name : ..\data\pdf_files\Arsenal Ebooks\Invincible - Amy Lawrence.pdf
Files Name : ..\data\pdf_files\Arsenal Ebooks\Stillness And Speed - Dennis Bergkamp.pdf
Files Name : ..\data\pdf_files\Arsenal Ebooks\The Romford Pele - Ray Parlour.pdf
Files Name : ..\data\pdf_files\Arsenal Ebooks\Thierry Henry - Phillipe Auclair.pdf


In [139]:
# select only pdf file
pdf_files = doc_dir.glob('*.pdf')
print(pdf_files)
pdf_files = list(pdf_files)
print(pdf_files)
for p in pdf_files:
    print(f'Files Name : {p}')

<generator object Path.glob at 0x0000024DD1043450>
[WindowsPath('../data/pdf_files/Arsenal Ebooks/Arsene Wenger - Tom Oldfield.pdf'), WindowsPath('../data/pdf_files/Arsenal Ebooks/Invincible - Amy Lawrence.pdf'), WindowsPath('../data/pdf_files/Arsenal Ebooks/Stillness And Speed - Dennis Bergkamp.pdf'), WindowsPath('../data/pdf_files/Arsenal Ebooks/The Romford Pele - Ray Parlour.pdf'), WindowsPath('../data/pdf_files/Arsenal Ebooks/Thierry Henry - Phillipe Auclair.pdf')]
Files Name : ..\data\pdf_files\Arsenal Ebooks\Arsene Wenger - Tom Oldfield.pdf
Files Name : ..\data\pdf_files\Arsenal Ebooks\Invincible - Amy Lawrence.pdf
Files Name : ..\data\pdf_files\Arsenal Ebooks\Stillness And Speed - Dennis Bergkamp.pdf
Files Name : ..\data\pdf_files\Arsenal Ebooks\The Romford Pele - Ray Parlour.pdf
Files Name : ..\data\pdf_files\Arsenal Ebooks\Thierry Henry - Phillipe Auclair.pdf


In [140]:
#Read PDF and Convert to Documents
documents = []
for pdf_file in pdf_files:
    loader = PyMuPDFLoader(str(pdf_file))
    docs = loader.load()
    metadata = {
        'createdBy' : 'Anshul',
        'createdDate' : '06-06-2026'
    }
    for doc in docs:
        #print(doc)
        #break
        doc.metadata.update(metadata)
    documents.extend(docs)



In [141]:
print(f'Number of documents generated : {len(documents)}')

Number of documents generated : 1277


In [142]:
#Chunking Process
textsplitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200,
    separators=['\n\n','\n',' ','']
)

In [143]:
chunks = textsplitter.split_documents(documents=documents)
print(f'Number of chunks created : {len(chunks)}')

Number of chunks created : 3720


In [144]:
# Creating Embeddings

st = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [145]:
text_ls = [ doc.page_content for doc in chunks]

In [146]:
text_ls[:3]

['For Noah and Melissa',
 'CONTENTS\nTitle Page\nDedication\nACKNOWLEDGEMENTS\nINTRODUCTION\nFROM PLAYER TO MANAGER\nARRIVING AT ARSENAL\nDOING THE DOUBLE AND LIVING IN THE SHADOW\nOF THE TREBLE WINNERS\nCHASING SIR ALEX’S COAT-TAILS\nSUPREMACY REGAINED AND SURRENDERED\nTHE INVINCIBLES AND THE SPECIAL ONE\nTHE REBUILDING PROCESS\nLIFE AFTER THIERRY\nADRIFT FROM THE PACK\nBACK IN CONTENTION\nON THE OUTSIDE LOOKING IN\nBOUNCING BACK FROM THE EXODUS\nTHE DROUGHT GOES ON\nSILVERWARE AT LAST\nFUTURE PLANS\nAPPENDIX: ARSÈNE’S ALL-TIME BEST ARSENAL\nXI\nCopyright',
 'ACKNOWLEDGEMENTS\nAnna Marx and the team at John Blake Publishing –\nThank you for your dedicated work in making this\nedition a reality. Your patience and diligence are\ngreatly appreciated.\nMy friends (KES, Nottingham and beyond) – Thank\nyou for your support and sense of humour. We\nwatched many of the games featured in this book\ntogether.\nMy work colleagues, past and present – You have\nbeen a priceless source of advice an

In [147]:
#create vectors
embeddings = st.encode(text_ls)

In [148]:
embeddings.shape

(3720, 384)

In [149]:
#Store documents in vector store
vectorclient = chromadb.PersistentClient(path = ".\\chroma.db")

In [150]:
print(vectorclient)

In [151]:
collection = vectorclient.get_or_create_collection(name = 'arsenal_db')

In [152]:
id_ls = [ f'{chunk.metadata.get('title','N/A')}__{i}' for i,chunk in enumerate(chunks,1) ]
print(id[10])

Arsene Wenger--The Unauthorised Biography of Le Professeur__11


In [153]:
chunks[10].metadata

{'producer': 'ToPDF.com',
 'creator': 'ToPDF.com',
 'creationdate': '2020-03-24T05:46:41+00:00',
 'source': '..\\data\\pdf_files\\Arsenal Ebooks\\Arsene Wenger - Tom Oldfield.pdf',
 'file_path': '..\\data\\pdf_files\\Arsenal Ebooks\\Arsene Wenger - Tom Oldfield.pdf',
 'total_pages': 193,
 'format': 'PDF 1.4',
 'title': 'Arsene Wenger--The Unauthorised Biography of Le Professeur',
 'author': 'Tom Oldfield',
 'subject': '',
 'keywords': '',
 'moddate': '2020-03-24T06:46:43+01:00',
 'trapped': '',
 'modDate': "D:20200324064643+01'00'",
 'creationDate': "D:20200324054641+00'00'",
 'page': 8,
 'createdBy': 'Anshul',
 'createdDate': '06-06-2026'}

In [154]:
metadata = [ doc.metadata for doc in chunks]

In [155]:
collection.add(
    ids = id_ls,
    embeddings = embeddings.tolist(),
    documents = text_ls, 
    metadatas =  metadata
)

In [240]:
# User Query
query = "Give me a paragraph of  250 words about Thierry Henry for arsenal?"

In [241]:
query_em = st.encode(query).tolist()

In [242]:
#Retrieve Relevant Chunks from vectordb

results = collection.query(query_embeddings=query_em,
                           n_results=3
                           )

In [243]:
print(results)

{'ids': [['Thierry Henry__3334', "Invincible: Inside Arsenal's Unbeaten 2003-2004 Season__1167", "Invincible: Inside Arsenal's Unbeaten 2003-2004 Season__1169"]], 'embeddings': None, 'documents': [['resist: a litany of goals doesn’t make for pleasant reading (or\nwriting, for that matter). Henry’s oeuvre is, in any case, only a\nclick away for everyone, and it makes more sense, once salient\nfacts had been given their due place in this narrative, to\nconcentrate on those events that reveal enough of the man and\nthe player to avoid the pitfalls of mere enumeration. I have\nmade an exception in the case of the 2002–3 season, however,\nprecisely because it demonstrated better than any previous or\nsubsequent campaign how Thierry had now reached a level of\nconsistency and excellence which he could maintain regardless\nof the dips in form that affected his team as a whole. It was\nperhaps the year in which he reached his absolute peak as an\nindividual player, whereas, for the Gunners, it

In [244]:
results.keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas', 'distances'])

In [245]:
type(results)

dict

In [246]:
results['documents']

[['resist: a litany of goals doesn’t make for pleasant reading (or\nwriting, for that matter). Henry’s oeuvre is, in any case, only a\nclick away for everyone, and it makes more sense, once salient\nfacts had been given their due place in this narrative, to\nconcentrate on those events that reveal enough of the man and\nthe player to avoid the pitfalls of mere enumeration. I have\nmade an exception in the case of the 2002–3 season, however,\nprecisely because it demonstrated better than any previous or\nsubsequent campaign how Thierry had now reached a level of\nconsistency and excellence which he could maintain regardless\nof the dips in form that affected his team as a whole. It was\nperhaps the year in which he reached his absolute peak as an\nindividual player, whereas, for the Gunners, it represented\nsomething of a stutter, bookended as it was by the Double of\n2002 and the remarkable run of forty-nine League games\nwithout defeat that was ended in contentious circumstances by',


In [247]:
#Create Prompt

system_prompt = """
You are an Ai Assistant for Arsenal and answer question on arsenal trivia. 
Answer only in the provided context.Be Precise and concise. 
If information is not in context , politely decline teh request
"""

context_text = """Context from documents :\n\n"""

for i,doc in enumerate(results['documents']):
    context_text += f'Document number : {i}\n{str(doc)}\n\n'


In [248]:
context_text

"Context from documents :\n\nDocument number : 0\n['resist: a litany of goals doesn’t make for pleasant reading (or\\nwriting, for that matter). Henry’s oeuvre is, in any case, only a\\nclick away for everyone, and it makes more sense, once salient\\nfacts had been given their due place in this narrative, to\\nconcentrate on those events that reveal enough of the man and\\nthe player to avoid the pitfalls of mere enumeration. I have\\nmade an exception in the case of the 2002–3 season, however,\\nprecisely because it demonstrated better than any previous or\\nsubsequent campaign how Thierry had now reached a level of\\nconsistency and excellence which he could maintain regardless\\nof the dips in form that affected his team as a whole. It was\\nperhaps the year in which he reached his absolute peak as an\\nindividual player, whereas, for the Gunners, it represented\\nsomething of a stutter, bookended as it was by the Double of\\n2002 and the remarkable run of forty-nine League games\\n

In [249]:
user_prompt = f""" 

{context_text}
Question : {query}
Answer:

"""

In [250]:
message = [
        {
            'role':'system',
            'content': system_prompt
        },
        {
            'role': 'user',
            'content' : user_prompt
        }
]

In [251]:
response = llm.invoke(
   message

)

In [252]:
answer = response.content

In [253]:
answer

'Thierry Henry is undoubtedly one of the most iconic players to have ever represented Arsenal FC. He arrived at the club in 1999, having previously played for Monaco and Juventus, but it was his passion for the team that truly set him apart from other international stars. Henry\'s love affair with Arsenal began when he watched Ian Wright\'s goals on a video message given to him by then-director David Dein, with Wright later stating that the relationship between them went beyond just being teammates, and that Thierry "fell in love" with the club because of him. This sentiment is echoed through Henry\'s own words, as he expressed his desire to learn about and become a part of the Arsenal history and legacy. As Henry settled into life at the Emirates Stadium, he proved himself to be an exemplary clubman, absorbing every aspect of the game that made it so special to the fans. His footballing prowess on the pitch was matched only by the impact he had on the dressing room, with teammates ali